# 07 — Analyse des erreurs TF-IDF (protocoles A et B)

**Lecture seule des expériences existantes : aucun entraînement, aucune nouvelle
inférence, aucun ajustement d'hyperparamètre.** Le notebook produit uniquement de
nouveaux tableaux d'analyse et le rapport `reports/tfidf_error_analysis.md`.
Les métriques proviennent des JSON existants ; la fonction commune `compute_metrics`
est utilisée seulement pour contrôler leur concordance en mémoire, sans écriture.

## 1. Configuration

Choisir le kernel `.venv`, puis Restart Kernel / Run All. Le dépôt est détecté depuis
la racine ou `notebooks/`. Sur Colab, monter le Drive puis renseigner le chemin
ci-dessous. Les exemples sont une sélection qualitative raisonnée, pas un échantillon
statistique représentatif de toutes les erreurs.

In [1]:
from pathlib import Path
import os
import sys
REPO_ROOT_OVERRIDE = None
ROOT = (Path(REPO_ROOT_OVERRIDE).expanduser().resolve() if REPO_ROOT_OVERRIDE is not None
        else next((p for p in [Path.cwd(), *Path.cwd().parents]
                   if (p/'README.md').is_file() and (p/'src/data.py').is_file()), None))
if ROOT is None:
    raise RuntimeError('Renseigner le chemin du dépôt dans REPO_ROOT_OVERRIDE.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
TABLE_DIR = ROOT/'results/tables'
REPORT_DIR = ROOT/'reports'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
import hashlib
import html
import json
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from PIL import Image
from src.data import EVENTS, LABELS, SEED, get_split
from src.evaluate import compute_metrics

# Seuls ces nouveaux livrables peuvent être réécrits lors d'une reproduction.
OUTPUT_NAMES = {'tfidf_error_analysis.csv', 'tfidf_error_analysis_all_errors.csv',
                'tfidf_error_analysis_A_classes.csv', 'tfidf_error_analysis_B_classes.csv',
                'tfidf_error_analysis_B_events.csv', 'tfidf_error_analysis_confusions.csv',
                'tfidf_error_analysis_transitions.csv', 'tfidf_error_analysis_checks.json'}
protected = set((ROOT/'splits').glob('*.csv')) | set((ROOT/'src').glob('*.py'))
for folder in (ROOT/'results', ROOT/'figures', ROOT/'notebooks', ROOT/'reports'):
    protected |= {p for p in folder.rglob('*') if p.is_file()
                  and p.name not in OUTPUT_NAMES and p.name not in
                  ('07_error_analysis_tfidf.ipynb','tfidf_error_analysis.md')}
protected_hashes = {str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest()
                    for p in sorted(protected)}
print('Fichiers existants protégés :', len(protected_hashes))

Fichiers existants protégés : 231


## 2. Validation des prédictions et jointure avec le texte original

Les populations viennent de `get_split` ; **aucun `all_target` n'est lu pour cette
analyse**. Chaque prédiction est vérifiée contre le test nettoyé : identifiants,
événement et vrai label doivent correspondre exactement. Probabilités finies,
ordre des classes, somme, argmax et probabilité de la classe prédite sont contrôlés.
Une « erreur » signifie un désaccord avec le label fourni, pas une preuve que
l'annotation ou le modèle est sémantiquement correct. La confiance est une
probabilité du modèle, pas une probabilité calibrée d'avoir raison.

In [2]:
IDENTITY = ['tweet_id','image_id','event_name','y_true']
checked_runs = []

def load_checked(task, test, prediction_relative, metric_relative):
    prediction_path, metric_path = ROOT/prediction_relative, ROOT/metric_relative
    p = pd.read_csv(prediction_path,dtype=str,keep_default_na=False,na_filter=False)
    labels = list(LABELS[task])
    probability_columns = ['proba_'+label for label in labels]
    assert list(p.columns) == IDENTITY[:3]+['y_true','y_pred']+probability_columns
    assert not p.duplicated(['tweet_id','image_id']).any()
    expected = test.rename(columns={'label':'y_true'})[IDENTITY]
    sort_key = ['tweet_id','image_id']
    pd.testing.assert_frame_equal(p[IDENTITY].sort_values(sort_key).reset_index(drop=True),
                                  expected.sort_values(sort_key).reset_index(drop=True))
    assert p.y_true.isin(labels).all() and p.y_pred.isin(labels).all()
    probabilities = p[probability_columns].to_numpy(float)
    assert np.isfinite(probabilities).all() and (probabilities>=0).all() and (probabilities<=1).all()
    assert np.allclose(probabilities.sum(axis=1),1,atol=1e-5,rtol=0)
    assert np.array_equal(np.asarray(labels)[probabilities.argmax(axis=1)],p.y_pred.to_numpy())
    p['predicted_class_probability'] = probabilities[np.arange(len(p)),[labels.index(v) for v in p.y_pred]]
    # Jointure univoque, texte brut et image repris tels quels du split.
    joined = p.merge(test[['tweet_id','image_id','tweet_text','image']],
                     on=['tweet_id','image_id'],how='left',validate='one_to_one',sort=False)
    assert joined.tweet_text.notna().all()
    metrics = json.loads(metric_path.read_text())
    assert metrics['task']==task and metrics['labels']==labels and metrics['n_samples']==len(p)
    assert metrics['subset']=='test'
    assert metrics['bootstrap']['n_resamples']==1000 and metrics['bootstrap']['seed']==SEED
    recalculated = compute_metrics(p.y_true,p.y_pred,task,n_bootstrap=1000,seed=SEED)
    for key in ('f1_macro','f1_weighted','accuracy'):
        assert np.isclose(metrics[key],recalculated[key],rtol=0,atol=1e-15)
    assert metrics['confusion_matrix']==recalculated['confusion_matrix']
    assert metrics['per_class']==recalculated['per_class']
    assert metrics['bootstrap']==recalculated['bootstrap']
    checked_runs.append(dict(task=task,predictions=prediction_relative,metrics=metric_relative,
                             rows=len(p),errors=int(p.y_true.ne(p.y_pred).sum())))
    return joined,metrics

A = {}
for task in LABELS:
    test = get_split(task,'A','official',text_only=True)['test']
    A[task] = load_checked(task,test,f'results/predictions/{task}_tfidf_A_official.csv',
                          f'results/metrics/{task}_tfidf_A_official.json')
B = {}
for task in LABELS:
    for event in EVENTS:
        test = get_split(task,'B',event,text_only=True)['test']
        stem = f'{task}_tfidf_B_{event}'
        domain_a,ma = load_checked(task,test,f'results/predictions/in_domain/{stem}.csv',
                                   f'results/metrics/in_domain/{stem}.json')
        domain_b,mb = load_checked(task,test,f'results/predictions/{stem}.csv',f'results/metrics/{stem}.json')
        pd.testing.assert_frame_equal(domain_a[IDENTITY],domain_b[IDENTITY])
        # Le sous-ensemble A est extrait de sorties existantes, jamais des données B.
        original = A[task][0].loc[A[task][0].event_name.eq(event)].set_index(['tweet_id','image_id'])
        aligned = original.loc[pd.MultiIndex.from_frame(domain_a[['tweet_id','image_id']])].reset_index()
        pd.testing.assert_frame_equal(aligned[IDENTITY+['y_pred']],domain_a[IDENTITY+['y_pred']])
        probability_columns = ['proba_'+label for label in LABELS[task]]
        np.testing.assert_array_equal(aligned[probability_columns].to_numpy(float),domain_a[probability_columns].to_numpy(float))
        B[task,event] = (domain_a,ma,domain_b,mb)
assert len(checked_runs)==30 and len(B)==14
print('30 CSV/JSON cohérents ; 14 comparaisons sur le test officiel uniquement.')

30 CSV/JSON cohérents ; 14 comparaisons sur le test officiel uniquement.


## 3. Protocole A — classes fragiles et confusions

Les scores et matrices sont lus dans les JSON. Les confusions sont dirigées
(vraie classe → prédiction), avec nombre, part des erreurs et part du support de
la vraie classe. Toutes les confusions, y compris les zéros, sont archivées ;
seules les plus fréquentes sont mises en avant. Aucun cas difficile n'est supprimé.

In [3]:
class_a_rows,confusion_rows,all_error_parts = [],[],[]
def collect_confusions(metrics,task,protocol,event,regime):
    matrix = np.asarray(metrics['confusion_matrix'])
    errors = matrix.sum()-np.trace(matrix)
    for i,truth in enumerate(metrics['labels']):
        for j,predicted in enumerate(metrics['labels']):
            if i!=j:
                count = int(matrix[i,j]); support = int(matrix[i].sum())
                confusion_rows.append(dict(task=task,protocol=protocol,event_name=event,regime=regime,
                                            y_true=truth,y_pred=predicted,count=count,support=support,
                                            share_of_errors=count/errors if errors else 0,
                                            share_of_true_class=count/support if support else np.nan))
for task,(frame,metrics) in A.items():
    for label,values in metrics['per_class'].items():
        class_a_rows.append(dict(task=task,label=label,**values))
    collect_confusions(metrics,task,'A','official','in_domain')
    errors = frame.loc[frame.y_true.ne(frame.y_pred)].copy()
    errors.insert(0,'task',task); errors.insert(1,'protocol','A')
    all_error_parts.append(errors)
classes_a = pd.DataFrame(class_a_rows)
classes_a.to_csv(TABLE_DIR/'tfidf_error_analysis_A_classes.csv',index=False)
for task in LABELS:
    print('Protocole A :',task)
    display(classes_a.loc[classes_a.task.eq(task)].sort_values(['f1-score','recall']))
    temp = pd.DataFrame(confusion_rows)
    display(temp.loc[temp.task.eq(task)&temp.protocol.eq('A')&temp['count'].gt(0)]
            .sort_values(['count','y_true','y_pred'],ascending=[False,True,True]).head(6))

Protocole A : informative


,task,label,precision,recall,f1-score,support
1,informative,not_informative,0.813333,0.605159,0.693970,504
0,informative,informative,0.828300,0.932039,0.877113,1030


,task,protocol,event_name,regime,y_true,y_pred,count,support,share_of_errors,share_of_true_class
1,informative,A,official,in_domain,not_informative,informative,199,504,0.739777,0.394841
0,informative,A,official,in_domain,informative,not_informative,70,1030,0.260223,0.067961


Protocole A : humanitarian


,task,label,precision,recall,f1-score,support
2,humanitarian,affected_individuals,0.375000,0.333333,0.352941,9
3,humanitarian,infrastructure_and_utility_damage,0.566667,0.629630,0.596491,81
4,humanitarian,rescue_volunteering_or_donation_effort,0.621118,0.793651,0.696864,126
5,humanitarian,other_relevant_information,0.672598,0.804255,0.732558,235
6,humanitarian,not_humanitarian,0.877108,0.722222,0.792165,504


,task,protocol,event_name,regime,y_true,y_pred,count,support,share_of_errors,share_of_true_class
21,humanitarian,A,official,in_domain,not_humanitarian,other_relevant_information,74,504,0.298387,0.146825
20,humanitarian,A,official,in_domain,not_humanitarian,rescue_volunteering_or_donation_effort,41,504,0.165323,0.081349
17,humanitarian,A,official,in_domain,other_relevant_information,not_humanitarian,23,235,0.092742,0.097872
19,humanitarian,A,official,in_domain,not_humanitarian,infrastructure_and_utility_damage,22,504,0.088710,0.043651
13,humanitarian,A,official,in_domain,rescue_volunteering_or_donation_effort,not_humanitarian,14,126,0.056452,0.111111
9,humanitarian,A,official,in_domain,infrastructure_and_utility_damage,not_humanitarian,12,81,0.048387,0.148148


## 4. Protocole B — dégradation par événement et par classe

Les transitions sont appariées : A correct/B faux, A faux/B correct, deux erreurs,
deux corrects. Un recul du F1 peut venir de faux positifs, même si le rappel augmente.
Les classes de support nul restent dans les fichiers, avec `support_zero=True` ;
elles ne sont pas classées comme « effondrées » en rappel. Les faibles supports sont
signalés, sans exclusion des résultats. Les IC des F1 macro sont ceux déjà enregistrés ;
aucun test de significativité ni IC du gap n'est ajouté.

In [4]:
class_b_rows,event_rows,transition_parts = [],[],[]
for (task,event),(a,ma,b,mb) in B.items():
    for label in LABELS[task]:
        sa,sb = ma['per_class'][label],mb['per_class'][label]
        assert sa['support']==sb['support']
        class_b_rows.append(dict(task=task,event_name=event,label=label,support=sa['support'],
                                 support_zero=sa['support']==0,small_support=0<sa['support']<10,
                                 f1_in_domain=sa['f1-score'],f1_out_of_domain=sb['f1-score'],
                                 f1_drop=sa['f1-score']-sb['f1-score'],
                                 recall_in_domain=sa['recall'],recall_out_of_domain=sb['recall'],
                                 recall_drop=sa['recall']-sb['recall'],
                                 precision_in_domain=sa['precision'],precision_out_of_domain=sb['precision']))
    transition = a[IDENTITY+['y_pred','predicted_class_probability']].merge(
        b[IDENTITY+['y_pred','predicted_class_probability']],on=IDENTITY,validate='one_to_one',suffixes=('_A','_B'))
    transition.insert(0,'task',task)
    transition['A_correct'] = transition.y_true.eq(transition.y_pred_A)
    transition['B_correct'] = transition.y_true.eq(transition.y_pred_B)
    transition_parts.append(transition)
    counts = transition.groupby(['A_correct','B_correct']).size()
    row = dict(task=task,event_name=event,n_test=len(b),
               f1_macro_in_domain=ma['f1_macro'],f1_macro_out_of_domain=mb['f1_macro'],
               generalization_gap=ma['f1_macro']-mb['f1_macro'],
               A_errors=int(a.y_true.ne(a.y_pred).sum()),B_errors=int(b.y_true.ne(b.y_pred).sum()),
               A_correct_B_wrong=int(counts.get((True,False),0)),A_wrong_B_correct=int(counts.get((False,True),0)),
               both_wrong=int(counts.get((False,False),0)),both_correct=int(counts.get((True,True),0)))
    for regime,metric in [('in_domain',ma),('out_of_domain',mb)]:
        for bound,value in metric['bootstrap']['intervals']['f1_macro'].items():
            row[f'f1_macro_{regime}_ci95_{bound}']=value
    event_rows.append(row)
    collect_confusions(ma,task,'B',event,'in_domain')
    collect_confusions(mb,task,'B',event,'out_of_domain')
    errors = b.loc[b.y_true.ne(b.y_pred)].copy()
    errors.insert(0,'task',task); errors.insert(1,'protocol','B')
    all_error_parts.append(errors)
classes_b = pd.DataFrame(class_b_rows)
events_b = pd.DataFrame(event_rows)
transitions = pd.concat(transition_parts,ignore_index=True)
confusions = pd.DataFrame(confusion_rows)
all_errors = pd.concat(all_error_parts,ignore_index=True)
classes_b.to_csv(TABLE_DIR/'tfidf_error_analysis_B_classes.csv',index=False)
events_b.to_csv(TABLE_DIR/'tfidf_error_analysis_B_events.csv',index=False)
transitions.to_csv(TABLE_DIR/'tfidf_error_analysis_transitions.csv',index=False)
confusions.to_csv(TABLE_DIR/'tfidf_error_analysis_confusions.csv',index=False)
all_errors.to_csv(TABLE_DIR/'tfidf_error_analysis_all_errors.csv',index=False)
# A et B couvrent exactement les mêmes paires test, chacun une fois par tâche.
assert len(transitions)==sum(len(frame) for frame,_ in A.values())==2489
assert len(classes_b)==49 and len(events_b)==14
assert len(all_errors)==sum(r['errors'] for r in checked_runs if '/in_domain/' not in r['predictions'])
display(events_b.sort_values('generalization_gap',ascending=False))
print('Dégradations F1 (support > 0 ; faibles supports conservés et signalés)')
display(classes_b.loc[~classes_b.support_zero].sort_values('f1_drop',ascending=False).head(15))
print('Toutes les erreurs A et B conservées :',len(all_errors))

,task,event_name,n_test,f1_macro_in_domain,f1_macro_out_of_domain,generalization_gap,A_errors,B_errors,A_correct_B_wrong,A_wrong_B_correct,both_wrong,both_correct,f1_macro_in_domain_ci95_lower,f1_macro_in_domain_ci95_upper,f1_macro_out_of_domain_ci95_lower,f1_macro_out_of_domain_ci95_upper
6,informative,srilanka_floods,94,0.897747,0.602444,0.295303,7,36,30,1,6,57,0.815586,0.962236,0.501405,0.695370
2,informative,hurricane_maria,359,0.818378,0.602241,0.216136,65,122,88,31,34,206,0.779604,0.857658,0.550076,0.654379
9,humanitarian,hurricane_maria,232,0.585073,0.395999,0.189074,48,117,74,5,43,110,0.527987,0.638569,0.325187,0.466073
11,humanitarian,iraq_iran_earthquake,23,0.523209,0.342857,0.180351,11,11,6,6,5,6,0.238886,0.722337,0.123077,0.476197
1,informative,hurricane_irma,324,0.717668,0.635272,0.082396,58,69,18,7,51,248,0.651626,0.776082,0.570433,0.701311
8,humanitarian,hurricane_irma,215,0.544671,0.480622,0.064048,61,76,20,5,56,134,0.485507,0.596621,0.411596,0.536809
7,humanitarian,hurricane_harvey,248,0.573364,0.524375,0.048989,62,71,20,11,51,166,0.514787,0.622813,0.464501,0.577028
10,humanitarian,mexico_earthquake,79,0.457991,0.416962,0.041029,31,34,7,4,27,41,0.346364,0.550641,0.294199,0.512445
4,informative,iraq_iran_earthquake,58,0.565000,0.527944,0.037056,12,15,3,0,12,43,0.422101,0.721154,0.402062,0.671202
3,informative,mexico_earthquake,127,0.610570,0.581530,0.029040,34,37,4,1,33,89,0.518939,0.702201,0.493416,0.672684


Dégradations F1 (support > 0 ; faibles supports conservés et signalés)


,task,event_name,label,support,support_zero,small_support,f1_in_domain,f1_out_of_domain,f1_drop,recall_in_domain,recall_out_of_domain,recall_drop,precision_in_domain,precision_out_of_domain
34,humanitarian,iraq_iran_earthquake,affected_individuals,4,False,True,0.600000,0.000000,0.600000,0.750000,0.000000,0.750000,0.500000,0.000000
37,humanitarian,iraq_iran_earthquake,other_relevant_information,5,False,True,0.545455,0.000000,0.545455,0.600000,0.000000,0.600000,0.500000,0.000000
5,informative,hurricane_maria,not_informative,165,False,False,0.808260,0.450450,0.357809,0.830303,0.303030,0.527273,0.787356,0.877193
28,humanitarian,hurricane_maria,not_humanitarian,129,False,False,0.844622,0.502793,0.341828,0.821705,0.348837,0.472868,0.868852,0.900000
12,informative,srilanka_floods,informative,20,False,False,0.844444,0.526316,0.318129,0.950000,1.000000,-0.050000,0.760000,0.357143
27,humanitarian,hurricane_maria,other_relevant_information,58,False,False,0.800000,0.488479,0.311521,0.896552,0.913793,-0.017241,0.722222,0.333333
13,informative,srilanka_floods,not_informative,74,False,False,0.951049,0.678571,0.272478,0.918919,0.513514,0.405405,0.985507,1.000000
25,humanitarian,hurricane_maria,infrastructure_and_utility_damage,21,False,False,0.628571,0.357143,0.271429,0.523810,0.238095,0.285714,0.785714,0.714286
45,humanitarian,srilanka_floods,infrastructure_and_utility_damage,3,False,True,0.800000,0.600000,0.200000,0.666667,1.000000,-0.333333,1.000000,0.428571
36,humanitarian,iraq_iran_earthquake,rescue_volunteering_or_donation_effort,1,False,True,0.500000,0.333333,0.166667,1.000000,1.000000,0.000000,0.333333,0.200000


Toutes les erreurs A et B conservées : 1256


## 5. Dix exemples sélectionnés et interprétation prudente

Sélection manuelle figée après lecture du texte : cinq exemples par tâche, six
relevant du protocole A et quatre du protocole B. Les six événements représentés,
les niveaux de confiance, les directions de confusion, les cas A correct/B faux,
A faux/B correct et les erreurs communes évitent de ne montrer qu'un type d'échec.
Cette sélection qualitative ne permet pas de mesurer la fréquence des causes.

Les catégories peuvent se recouper. Un échec sur un texte explicite peut rester
**non attribué** : nous n'inventons pas une explication. Les remarques sur les labels
sont des pistes de relecture, pas des corrections. Les images sont affichées depuis
les fichiers réels pour la vérification ; les hypothèses ci-dessous portent sur
le texte, sans audit d'annotation multimodale ni preuve sur les poids TF-IDF.

In [5]:
# Identifiants d'exemples éditoriaux, jamais utilisés pour ajuster les modèles.
SELECTION = [
 dict(task='informative',protocol='A',tweet_id='921578610624000004',
      category='Texte ambigu ou trop court; Texte qui ne contient pas assez d’informations pour identifier la classe',
      observation='Le texte mentionne une collecte par le père pour Puerto Rico, sans détailler les objets ni les destinataires.',
      hypothesis='Un message informel et peu explicite peut masquer la dimension d’aide ; le rôle des mots géographiques n’est pas établi.',
      reason='Faux négatif assez confiant, forme informelle d’aide.'),
 dict(task='informative',protocol='A',tweet_id='923475560029974528',
      category='Texte qui ne contient pas assez d’informations pour identifier la classe',
      observation='Le texte visible contient essentiellement RT, un nom et deux liens.',
      hypothesis='Le texte seul ne décrit pas la catastrophe ni une action humanitaire ; cette forte probabilité erronée ne prouve pas une cause lexicale particulière.',
      reason='Faux positif très confiant avec presque aucun contenu exploitable.'),
 dict(task='informative',protocol='A',tweet_id='905664032949993472',
      category='Texte ambigu ou trop court',
      observation='Question rhétorique sur Harvey, Irma et les États climatosceptiques ; la cible fournie est informative.',
      hypothesis='La frontière entre commentaire politique et information utile est ambiguë ici ; aucun label n’est changé. B classe ce cas correctement, contrairement à A.',
      reason='Décision proche du seuil et contre-exemple A faux/B correct.'),
 dict(task='informative',protocol='B',tweet_id='871037159326830592',
      category='Vocabulaire spécifique à une catastrophe ou à un lieu',
      observation='Le texte cite Sadrith Mora, #ESO et #Morrowind, mais est rattaché à l’événement srilanka_floods.',
      hypothesis='Le nom Mora apparaît dans un contexte de jeu vidéo ; une association lexicale hors contexte est plausible, sans preuve qu’il a piloté la décision du modèle.',
      reason='Faux positif hors-domaine très confiant, A correct.'),
 dict(task='informative',protocol='B',tweet_id='920348743316180993',
      category='Texte qui ne contient pas assez d’informations pour identifier la classe',
      observation='Le tweet demande comment les incendies affecteront le vin et renvoie à un reportage, sans donner la réponse.',
      hypothesis='La portée informative peut dépendre du contenu lié ; le texte seul ne permet pas de vérifier cette portée. Ce cas échoue sous B malgré une amélioration moyenne sur California.',
      reason='Erreur de confiance modérée dans un événement où B améliore le F1 global.'),
 dict(task='humanitarian',protocol='A',tweet_id='905221184819781632',
      category='Non attribuée à partir du texte',
      observation='Une enfant de dix jours et sa famille sont explicitement décrites comme secourues après Harvey.',
      hypothesis='Le texte contient des indices humanitaires explicites ; la prédiction not_humanitarian reste inexpliquée ici. Ce cas contredit une explication générale par manque d’information.',
      reason='Échec difficile sur la classe rare affected_individuals, sans explication forcée.'),
 dict(task='humanitarian',protocol='A',tweet_id='912013825863557121',
      category='Confusion entre deux classes proches',
      observation='Le texte mentionne des dégâts sur le principal hôpital de Dominica et des conditions de zone de guerre.',
      hypothesis='Le dommage à une infrastructure et l’information générale sur la catastrophe coexistent ; le modèle choisit la catégorie générale, sans preuve de son mécanisme interne.',
      reason='Confusion infrastructure → other, confiance intermédiaire.'),
 dict(task='humanitarian',protocol='A',tweet_id='911846702558138368',
      category='Erreur potentielle d’annotation ou étiquette discutable',
      observation='Le texte décrit des volontaires remplissant une semi-remorque pour les secours du séisme au Mexique ; le label fourni est not_humanitarian.',
      hypothesis='Le sens textuel paraît compatible avec la classe prédite rescue ; une relecture de l’annotation et du contexte multimodal serait nécessaire. Ce n’est pas une preuve d’erreur d’annotation.',
      reason='Label discutable et erreur très confiante, conservée comme erreur officielle.'),
 dict(task='humanitarian',protocol='B',tweet_id='910579624857522176',
      category='Confusion entre deux classes proches; Vocabulaire spécifique à une catastrophe ou à un lieu',
      observation='Le texte décrit des maisons endommagées et des toits arrachés aux USVI par Maria.',
      hypothesis='La description des dégâts est explicite ; le contexte local peut accompagner le décalage, mais on ne peut pas l’isoler comme cause. A est correct et B prédit other.',
      reason='Dégradation A correct/B faux dans une classe touchée sur Maria.'),
 dict(task='humanitarian',protocol='B',tweet_id='930293724990410752',
      category='Confusion entre deux classes proches',
      observation='Le texte associe 450 morts, 7 000 blessés et la fin des missions de secours en Iran.',
      hypothesis='Les indices de victimes et de secours rendent deux catégories sémantiquement plausibles ; le label officiel affected_individuals reste la référence.',
      reason='Rappel nul de la classe affectée en Irak-Iran sous B, exemple de confiance modérée.'),
]
selected_rows=[]
for index,item in enumerate(SELECTION,start=1):
    task,protocol,tweet_id = item['task'],item['protocol'],item['tweet_id']
    official = A[task][0]
    source_a = official.loc[official.tweet_id.eq(tweet_id)]
    assert len(source_a)==1
    event = source_a.iloc[0].event_name
    a,_,b,_ = B[task,event]
    source = official if protocol=='A' else b
    row = source.loc[source.tweet_id.eq(tweet_id)].iloc[0]
    assert row.y_true!=row.y_pred
    counterpart_a = a.loc[a.tweet_id.eq(tweet_id)].iloc[0]
    counterpart_b = b.loc[b.tweet_id.eq(tweet_id)].iloc[0]
    image_path = ROOT/row.image
    image_exists = image_path.is_file()
    if image_exists:
        with Image.open(image_path) as im:
            im.load()
    selected_rows.append(dict(example_id=f'E{index:02}',task=task,protocol=protocol,
                              tweet_id=row.tweet_id,image_id=row.image_id,event_name=row.event_name,
                              tweet_text=row.tweet_text,y_true=row.y_true,y_pred=row.y_pred,
                              confidence=float(row.predicted_class_probability),
                              image=row.image,image_exists=image_exists,
                              error_type=item['category'],observation=item['observation'],
                              hypothesis=item['hypothesis'],interpretation_status='Hypothèse qualitative, non causale',
                              selection_reason=item['reason'],
                              y_pred_A=counterpart_a.y_pred,y_pred_B=counterpart_b.y_pred,
                              confidence_A=float(counterpart_a.predicted_class_probability),
                              confidence_B=float(counterpart_b.predicted_class_probability),
                              A_correct=counterpart_a.y_true==counterpart_a.y_pred,
                              B_correct=counterpart_b.y_true==counterpart_b.y_pred,
                              prediction_source=f'results/predictions/{task}_tfidf_A_official.csv' if protocol=='A'
                                                else f'results/predictions/{task}_tfidf_B_{event}.csv'))
selected = pd.DataFrame(selected_rows)
assert len(selected)==10 and selected.groupby('task').size().eq(5).all()
selected.to_csv(TABLE_DIR/'tfidf_error_analysis.csv',index=False)
for item in selected_rows:
    display(HTML(f'<h3>{item["example_id"]} · {html.escape(item["task"])} · {item["protocol"]}</h3>'
                 f'<p>ID : {item["tweet_id"]} · {html.escape(item["event_name"])}<br>'
                 f'Vrai : {html.escape(item["y_true"])} → prédit : {html.escape(item["y_pred"])}<br>'
                 f'Probabilité de la classe prédite : {item["confidence"]:.4f}</p>'
                 f'<pre style="white-space:pre-wrap">{html.escape(item["tweet_text"])}</pre>'
                 f'<p><b>Constat :</b> {html.escape(item["observation"])}<br>'
                 f'<b>Hypothèse :</b> {html.escape(item["hypothesis"])}<br>'
                 f'<b>Image :</b> {html.escape(item["image"])}</p>'))
    if item['image_exists']:
        with Image.open(ROOT/item['image']) as im:
            preview=im.convert('RGB')
            preview.thumbnail((480,320))
            display(preview)
    else:
        print('Image indisponible : aucun contenu inventé.')

## 6. Rapport pour le milestone et le rapport final

Les observations quantitatives sont générées depuis les métriques vérifiées.
Les annotations éditoriales sont figées dans la cellule précédente ; elles restent
séparées du texte original et de la vérité fournie. Le rapport donne les résultats
A et B dans des sections distinctes et garde explicitement les améliorations B.

In [6]:
def markdown_table(frame,columns):
    def safe(value):
        if isinstance(value,float):
            return f'{value:.4f}'
        return str(value).replace('|','\|').replace('\n','<br>')
    lines=['| '+' | '.join(columns)+' |','|'+'|'.join(['---']*len(columns))+'|']
    lines += ['| '+' | '.join(safe(row[c]) for c in columns)+' |' for row in frame.to_dict('records')]
    return '\n'.join(lines)

lines=['# Analyse des erreurs de la baseline TF-IDF — DATA4001','',
       '## Périmètre et méthode','',
       'Analyse des prédictions existantes, sans entraînement, inférence, modification de split, de métrique ou d’hyperparamètre. '
       'Les vrais labels sont ceux des CSV nettoyés agreed_lab. Les jointures vérifient exactement tweet_id, image_id, event_name et y_true. '
       'Les 30 ensembles de prédictions/métriques lus (2 A et 28 sous-ensembles B in/out) sont contrôlés en mémoire avec la fonction commune compute_metrics. '
       'Aucun all_target ne sert à la comparaison A/B. Les probabilités correspondent aux colonnes des classes prédites ; elles ne sont pas nécessairement calibrées.','',
       'Les dix exemples sont une sélection qualitative raisonnée : cinq par tâche, six A et quatre B, six événements, plusieurs directions de confusion et niveaux de confiance. '
       'Ils ne constituent pas un échantillon statistique représentatif des causes. Toutes les erreurs, y compris celles non sélectionnées, restent dans '
       'results/tables/tfidf_error_analysis_all_errors.csv ; les transitions de toutes les paires sont également conservées.','',
       '## 1. Protocole A — observations quantitatives','']
for task,(frame,metrics) in A.items():
    errors=int(frame.y_true.ne(frame.y_pred).sum())
    lines += [f'### {task}', '', f'{errors} erreurs sur {len(frame)} lignes ; F1 macro {metrics["f1_macro"]:.4f}.', '',
              markdown_table(classes_a.loc[classes_a.task.eq(task)].sort_values('f1-score'),
                             ['label','precision','recall','f1-score','support']),'',
              'Principales confusions dirigées :','',
              markdown_table(confusions.loc[confusions.task.eq(task)&confusions.protocol.eq('A')&confusions['count'].gt(0)]
                             .sort_values('count',ascending=False).head(6),
                             ['y_true','y_pred','count','support','share_of_errors','share_of_true_class']),'']
lines += ['En informative, not_informative a le F1 et le rappel les plus faibles. Les faux positifs de la classe informative dominent. '
          'En humanitarian, affected_individuals est la classe la plus fragile, mais seulement 9 exemples test sont disponibles : '
          '3 correctement retrouvés, 6 manqués. Les erreurs de not_humanitarian vers other_relevant_information et rescue sont '
          'les confusions les plus nombreuses en valeur absolue ; cela ne signifie pas que cette classe ait le plus mauvais taux d’erreur.','',
          '## 2. Protocole B — généralisation sur le test officiel','',
          'Les F1 A et B ci-dessous portent sur exactement les mêmes paires de chaque événement. '
          'Un gap positif favorise A ; un gap négatif favorise B. Les IC95 par modèle figurent dans le CSV détaillé, '
          'sans être interprétés comme un IC du gap.','',
          markdown_table(events_b,['task','event_name','n_test','f1_macro_in_domain','f1_macro_out_of_domain',
                                   'generalization_gap','A_correct_B_wrong','A_wrong_B_correct']),'',
          '### Classes qui se dégradent le plus (support non nul)','',
          markdown_table(classes_b.loc[~classes_b.support_zero].sort_values('f1_drop',ascending=False).head(12),
                         ['task','event_name','label','support','f1_in_domain','f1_out_of_domain','f1_drop',
                          'recall_in_domain','recall_out_of_domain']),'',
          'Sur Maria, not_informative perd fortement en rappel et not_humanitarian est davantage envoyé vers other_relevant_information. '
          'Le F1 de other_relevant_information baisse aussi alors que son rappel augmente légèrement : la précision se dégrade, '
          'donc il ne faut pas décrire cette baisse de F1 comme un effondrement du rappel.','',
          'Sur srilanka_floods, le modèle B informative prédit informative pour des messages non informatifs : '
          'le rappel de informative augmente, mais son F1 diminue sous l’effet des faux positifs. '
          'En humanitarian Irak-Iran, affected_individuals et other_relevant_information ont un rappel nul sous B, '
          'sur seulement 4 et 5 exemples. Les 23 lignes totales et les IC larges imposent une lecture prudente.','',
          'California est un contre-exemple à une dégradation systématique du F1 macro : il est meilleur sous B que sous A. '
          'Ce gain ne correspond pas à moins d’erreurs : elles passent de 31 à 39 (13 nouveaux échecs contre 5 corrections). '
          'Le rappel de not_informative passe de 10,34 % à 27,59 %, tandis que celui de informative recule de 96,67 % à 88,00 %. '
          'Le F1 macro pondère également les classes, contrairement au nombre brut d’erreurs : ces deux observations sont compatibles. '
          'Les transitions A faux/B correct sont rapportées pour tous les événements, et E03 en montre un cas. '
          'E05 reste une erreur sous B en California malgré ce gain moyen.','',
          '### Principales confusions B pour chaque tâche et événement','']
for task in LABELS:
    for event in EVENTS:
        top=confusions.loc[confusions.task.eq(task)&confusions.protocol.eq('B')&confusions.event_name.eq(event)
                            &confusions.regime.eq('out_of_domain')&confusions['count'].gt(0)]
        lines += [f'**{task} / {event}**','',
                  markdown_table(top.sort_values('count',ascending=False).head(3),
                                 ['y_true','y_pred','count','support','share_of_true_class']),'']
lines += ['## 3. Dix exemples vérifiables et catégories qualitatives','',
          'Les constats décrivent ce qui est lisible dans le texte. Les hypothèses ne démontrent ni le mécanisme TF-IDF ni '
          'une erreur d’annotation. Aucune cause n’est imposée lorsqu’elle ne se déduit pas du texte. '
          'Une image réelle est disponible pour chaque exemple sélectionné et peut être affichée dans le notebook ; '
          'les commentaires ci-dessous ne prétendent pas avoir établi son contenu ou refait l’annotation multimodale.','']
for r in selected_rows:
    lines += [f'### {r["example_id"]} — {r["task"]}, protocole {r["protocol"]}', '',
              f'- Tweet : `{r["tweet_id"]}` ; image : `{r["image_id"]}` ; événement : `{r["event_name"]}`.',
              f'- Vraie classe : `{r["y_true"]}` ; prédite : `{r["y_pred"]}` ; confiance : **{r["confidence"]:.4f}**.',
              f'- Comparaison : A → `{r["y_pred_A"]}` (p={r["confidence_A"]:.4f}), B → `{r["y_pred_B"]}` (p={r["confidence_B"]:.4f}).',
              f'- Image locale : `{r["image"]}` ; fichier présent : {r["image_exists"]}.','',
              'Texte original (aucune normalisation) :','', '<pre>'+html.escape(r['tweet_text'])+'</pre>', '',
              f'**Catégorie :** {r["error_type"]}.', '',f'**Observation :** {r["observation"]}', '',
              f'**Hypothèse, non démontrée :** {r["hypothesis"]}', '',f'**Intérêt de la sélection :** {r["selection_reason"]}','']
lines += ['## 4. Texte réutilisable pour les livrables','',
          '### Milestone — constat principal','',
          'Sur le test officiel, la baseline TF-IDF réalise 269 erreurs en informative et 248 en humanitarian. '
          'Elle retrouve difficilement les messages not_informative (rappel 60,52 %) et la classe rare affected_individuals '
          '(rappel 33,33 %, F1 35,29 %, n=9). Les exemples illustrent des textes peu explicites, des frontières entre '
          'catégories proches et des désaccords d’annotation possibles, sans permettre une attribution causale certaine.','',
          '### Rapport final — généralisation','',
          'La comparaison appariée sur le test officiel montre des pertes de F1 macro importantes en informative pour '
          'srilanka_floods (29,53 points) et hurricane_maria (21,61 points), et en humanitarian pour hurricane_maria '
          '(18,91 points). Ces pertes ne sont pas systématiques : informative/California gagne 4,83 points hors-domaine. '
          'L’inspection par classe distingue la perte de rappel de la contamination par faux positifs ; les faibles '
          'supports de certaines classes et les intervalles de confiance empêchent de généraliser ces constats sans réserve.','',
          '## 5. Vérifications et limites','',
          '- 30 fichiers de prédictions et leurs JSON validés ; vrais labels identiques aux splits et probabilités de la classe prédite vérifiées.',
          '- 14 comparaisons A/B sur le test officiel, jamais sur all_target ; prédictions in-domain identiques aux sorties A originales.',
          '- Tous les cas erronés A et B conservés dans l’inventaire complet ; les exemples ne servent pas à estimer la fréquence des catégories qualitatives.',
          '- Les classes absentes restent dans les métriques de la tâche avec la convention du module commun ; elles sont signalées et ne sont pas traitées comme des pertes de rappel sur des exemples observés.',
          '- Les noms de lieux et indices lexicaux suggèrent des hypothèses de décalage ; aucun poids de modèle ni ablation ne démontre ici un mécanisme causal.',
          '- Les étiquettes discutables sont seulement signalées pour une relecture future ; aucun label n’est modifié.',
          '- Aucun entraînement, nouvelle prédiction, réglage sur le test ou modification d’un résultat existant.',
          '- Images : présence et décodage contrôlés pour les dix exemples, sans assertion inventée sur leur contenu.',
          '- Empreintes des entrées et résultats préexistants vérifiées en fin d’exécution ; contrôles dans results/tables/tfidf_error_analysis_checks.json.', '',
          '## 6. Fichiers produits','',
          '- notebooks/07_error_analysis_tfidf.ipynb : analyse exécutée, textes et images réelles affichés.',
          '- results/tables/tfidf_error_analysis.csv : dix exemples et annotations qualitatives.',
          '- results/tables/tfidf_error_analysis_all_errors.csv : inventaire exhaustif A + B hors-domaine sur test officiel.',
          '- results/tables/tfidf_error_analysis_A_classes.csv et tfidf_error_analysis_B_classes.csv : scores et variations par classe.',
          '- results/tables/tfidf_error_analysis_B_events.csv : comparaison par événement, IC95 et transitions.',
          '- results/tables/tfidf_error_analysis_confusions.csv : toutes les confusions dirigées A et B in/out.',
          '- results/tables/tfidf_error_analysis_transitions.csv : les 2 489 comparaisons appariées, succès compris.',
          '- results/tables/tfidf_error_analysis_checks.json : validation et empreintes.',
          '- reports/tfidf_error_analysis.md : présent rapport.', '']
(REPORT_DIR/'tfidf_error_analysis.md').write_text('\n'.join(lines),encoding='utf-8')
for relative,expected in protected_hashes.items():
    assert hashlib.sha256((ROOT/relative).read_bytes()).hexdigest()==expected,relative
# Vérification de la sérialisation du texte, des IDs et des probabilités sélectionnées.
round_trip=pd.read_csv(TABLE_DIR/'tfidf_error_analysis.csv',dtype=str,keep_default_na=False)
assert round_trip.tweet_text.tolist()==selected.tweet_text.tolist()
assert round_trip.tweet_id.tolist()==selected.tweet_id.tolist()
np.testing.assert_array_equal(round_trip.confidence.to_numpy(float),selected.confidence.to_numpy(float))
checks=dict(status='passed',checked_prediction_metric_pairs=30,official_test_B_comparisons=14,
            selected_examples=10,full_error_rows=len(all_errors),paired_test_rows=len(transitions),
            selected_images_present=int(selected.image_exists.sum()),
            checks=['true_labels_match_cleaned_splits','probability_matches_predicted_class',
                    'prediction_argmax','existing_metrics_match_shared_compute_metrics',
                    'A_B_identical_test_populations','A_predictions_reused_exactly',
                    'all_target_not_used','all_errors_preserved','selected_original_text_round_trip',
                    'existing_artifacts_sha256_unchanged'],protected_sha256=protected_hashes)
(TABLE_DIR/'tfidf_error_analysis_checks.json').write_text(json.dumps(checks,indent=2)+'\n')
print('Analyse terminée : 10 exemples,',len(all_errors),'erreurs archivées, 2 489 comparaisons ; entrées inchangées.')

Analyse terminée : 10 exemples, 1256 erreurs archivées, 2 489 comparaisons ; entrées inchangées.
